# Dense / Embedding Ablation

Kaggle-ready notebook for running dense embedding ablations from prebuilt FAISS indexes.

Use this when `index.faiss` and `payloads.jsonl` already exist for:

- `Embed-ChunkMeta-Dense`
- `Embed-ChunkOnly-Dense`

This notebook builds `payload_cache.sqlite` automatically when missing, runs the same QA benchmark for both indexes, and writes comparable retrieval metrics + latency.


In [1]:
!nvidia-smi || true
!pip -q install faiss-cpu sentence-transformers pandas tqdm numpy

%cd /kaggle/working
!test -d TextMining || git clone https://github.com/PhuongThao-2005/TextMining.git

Mon Jul 27 17:33:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path('/kaggle/working/TextMining')
if not (PROJECT_ROOT / 'src').exists():
    raise FileNotFoundError(f'Repo src not found at {PROJECT_ROOT / "src"}. Check git clone output above.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('src exists   =', (PROJECT_ROOT / 'src').exists())

PROJECT_ROOT = /kaggle/working/TextMining
src exists   = True


In [3]:
from pathlib import Path

# Kaggle input paths for the current uploaded datasets.

CHUNK_META_INDEX_DIR = Path('/kaggle/input/datasets/kittrntunk/faiss-chunk-meta')
CHUNK_ONLY_INDEX_DIR = Path('/kaggle/input/datasets/kittrntunk/faisse-only-chunk/faiss_index')
CHUNK_SIZE_FAISS_ROOT = Path('/kaggle/input/datasets/kittrntunk/chunk-size-faiss-indexes')
WORK_INDEX_ROOT = Path('/kaggle/working/faiss_indexes')

CONFIGS = {
    'Embed-ChunkMeta-Dense': CHUNK_META_INDEX_DIR,
    'Embed-ChunkOnly-Dense': CHUNK_ONLY_INDEX_DIR,
}

QA_PATH = Path('/kaggle/input/datasets/phuongthao205/qa-legalrag/Benchmark/qa_final.jsonl')

MODEL_NAME = 'intfloat/multilingual-e5-large'
TOP_K_LIST = [1, 5, 10]
RETRIEVE_K = max(TOP_K_LIST)
SAMPLE_LIMIT = None  # set None for full benchmark
OUT_DIR = Path('/kaggle/working/evaluation_runs/dense_embedding_ablation')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('QA_PATH =', QA_PATH, '| exists:', QA_PATH.exists())
for name, path in CONFIGS.items():
    print(name, '->', path, '| exists:', path.exists())

QA_PATH = /kaggle/input/datasets/phuongthao205/qa-legalrag/Benchmark/qa_final.jsonl | exists: True
Embed-ChunkMeta-Dense -> /kaggle/input/datasets/kittrntunk/faiss-chunk-meta | exists: True
Embed-ChunkOnly-Dense -> /kaggle/input/datasets/kittrntunk/faisse-only-chunk/faiss_index | exists: True


In [4]:
import os, shutil, sqlite3

def link_or_copy(src, dest):
    if dest.exists():
        return
    try:
        os.symlink(src, dest)
    except Exception:
        shutil.copy2(src, dest)

def copy_cache_and_refresh_meta(src_cache, dest_cache, payloads_path):
    # Cache meta stores payload size + mtime. Kaggle dataset mounts can change
    # mtimes between uploads, so refresh the copied cache meta to match the
    # current payloads.jsonl and avoid rebuilding a valid cache.
    if not src_cache.exists() or dest_cache.exists():
        return
    shutil.copy2(src_cache, dest_cache)
    stat = payloads_path.stat()
    conn = sqlite3.connect(str(dest_cache))
    try:
        conn.execute("UPDATE meta SET value=? WHERE key='payload_size'", (str(stat.st_size),))
        conn.execute("UPDATE meta SET value=? WHERE key='payload_mtime_ns'", (str(stat.st_mtime_ns),))
        conn.commit()
    finally:
        conn.close()

def prepare_writable_index_dir(config_name, src_dir):
    # Kaggle input is read-only. Create a tiny writable mirror with symlinks to
    # the large FAISS/payload files. If payload_cache.sqlite is missing, it will
    # be created in this writable directory without duplicating index.faiss.
    dest = WORK_INDEX_ROOT / config_name
    dest.mkdir(parents=True, exist_ok=True)
    for filename in ['index.faiss', 'payloads.jsonl', 'id_map.json', 'payload_offsets.pkl', 'payloads_export.csv']:
        src = src_dir / filename
        if src.exists():
            link_or_copy(src, dest / filename)
    copy_cache_and_refresh_meta(src_dir / 'payload_cache.sqlite', dest / 'payload_cache.sqlite', dest / 'payloads.jsonl')
    return dest

RUNTIME_INDEX_DIRS = {}
for config_name, src_dir in CONFIGS.items():
    missing = [name for name in ['index.faiss', 'payloads.jsonl'] if not (src_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f'{config_name} missing required files: {missing} under {src_dir}')
    RUNTIME_INDEX_DIRS[config_name] = prepare_writable_index_dir(config_name, src_dir)

RUNTIME_INDEX_DIRS

{'Embed-ChunkMeta-Dense': PosixPath('/kaggle/working/faiss_indexes/Embed-ChunkMeta-Dense'),
 'Embed-ChunkOnly-Dense': PosixPath('/kaggle/working/faiss_indexes/Embed-ChunkOnly-Dense')}

In [5]:
import json, time, math
from datetime import datetime, timezone
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

from retrieval.sqlite_faiss_store import SQLitePayloadFaissVectorStore

try:
    from evaluation.metrics import mrr, ndcg_at_k
except Exception:
    mrr = ndcg_at_k = None

def read_jsonl(path, limit=None):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            if limit is not None and len(rows) >= limit:
                break
    return rows

qa_rows = read_jsonl(QA_PATH, SAMPLE_LIMIT)
print('QA rows:', len(qa_rows))
print('Sample keys:', qa_rows[0].keys() if qa_rows else None)

QA rows: 500
Sample keys: dict_keys(['qa_id', 'question', 'reference_answer', 'answer_explanation', 'answer_type', 'ground_truth', 'edges_used', 'category', 'difficulty', 'source_type', 'generator_model', 'corpus_version', 'as_of_date', 'status', 'source_bundle_id', 'source_plan_id', 'center_provision_id', 'persona_used', 'verifier_model', 'verifier_result', 'verifier_checks', 'verifier_notes'])


In [6]:
def get_question(row):
    return row.get('question') or row.get('query') or row.get('input') or ''

def ground_truth_chunk_ids(row):
    gt = row.get('ground_truth') or {}
    ids = gt.get('chunk_ids') or row.get('ground_truth_chunk_ids') or row.get('relevant_chunk_ids') or []
    if isinstance(ids, str):
        return {ids}
    return {str(x) for x in ids if x is not None}

def dcg(binary_relevance):
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(binary_relevance))

def metrics_for_case(hit_chunk_ids, gt_ids, top_k_list):
    out = {}
    if not gt_ids:
        for k in top_k_list:
            out[f'recall@{k}'] = None
            out[f'ndcg@{k}'] = None
        out['mrr@10'] = None
        return out

    for k in top_k_list:
        top = hit_chunk_ids[:k]
        out[f'recall@{k}'] = len(set(top) & gt_ids) / len(gt_ids)
        rel = [1 if cid in gt_ids else 0 for cid in top]
        ideal = [1] * min(len(gt_ids), k)
        out[f'ndcg@{k}'] = dcg(rel) / dcg(ideal) if ideal else 0.0

    rr = 0.0
    for rank, cid in enumerate(hit_chunk_ids[:10], start=1):
        if cid in gt_ids:
            rr = 1.0 / rank
            break
    out['mrr@10'] = rr
    return out

def aggregate_cases(cases):
    metric_keys = sorted({k for row in cases for k, v in row.items() if k.startswith(('recall@', 'ndcg@')) or k == 'mrr@10'})
    summary = {'n_cases': len(cases)}
    for key in metric_keys:
        vals = [row[key] for row in cases if row.get(key) is not None]
        summary[key] = float(np.mean(vals)) if vals else None
    for key in ['embed_ms', 'search_ms', 'total_ms']:
        vals = [row[key] for row in cases if row.get(key) is not None]
        summary[f'{key}_mean'] = float(np.mean(vals)) if vals else None
        summary[f'{key}_median'] = float(np.median(vals)) if vals else None
    return summary

def aggregate_by_field(cases, field):
    groups = defaultdict(list)
    for row in cases:
        groups[str(row.get(field) or 'UNKNOWN')].append(row)
    return {name: aggregate_cases(rows) for name, rows in sorted(groups.items())}

def latency_summary(summary):
    return {
        'embed_ms_mean': summary.get('embed_ms_mean'),
        'embed_ms_median': summary.get('embed_ms_median'),
        'search_ms_mean': summary.get('search_ms_mean'),
        'search_ms_median': summary.get('search_ms_median'),
        'total_ms_mean': summary.get('total_ms_mean'),
        'total_ms_median': summary.get('total_ms_median'),
    }

def write_report(run_dir, config_name, summary, breakdowns, manifest):
    lines = [
        f'# {config_name}',
        '',
        '## Manifest',
        '',
        f"- run_id: `{manifest['run_id']}`",
        f"- benchmark_path: `{manifest['benchmark_path']}`",
        f"- index_path: `{manifest['index_path']}`",
        f"- model: `{manifest['model']}`",
        f"- n_evaluated: {manifest['n_evaluated']}",
        '',
        '## Retrieval Metrics',
        '',
        '| Metric | Value |',
        '| --- | ---: |',
    ]
    for key in sorted(summary):
        if key == 'n_cases':
            continue
        value = summary[key]
        value_text = '' if value is None else f'{value:.6f}' if isinstance(value, float) else str(value)
        lines.append(f'| `{key}` | {value_text} |')
    lines.extend(['', '## Breakdown Files', '', '- `retrieval_breakdown.json`', '- `latency.json`'])
    (run_dir / 'report.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')

In [7]:
model = SentenceTransformer(MODEL_NAME)
embed_dim = model.get_sentence_embedding_dimension()
print('model:', MODEL_NAME)
print('dim:', embed_dim)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

model: intfloat/multilingual-e5-large
dim: 1024


/tmp/ipykernel_24/3425533312.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_dim = model.get_sentence_embedding_dimension()


In [8]:
def run_config(config_name, index_dir):
    print('\n' + '=' * 80)
    print(config_name)
    print('index_dir:', index_dir)
    store = SQLitePayloadFaissVectorStore.load(index_dir)
    if store.total_vectors != sum(1 for _ in open(index_dir / 'payloads.jsonl', encoding='utf-8')):
        raise AssertionError(f'FAISS/payload mismatch for {config_name}')
    print('vectors:', store.total_vectors)

    cases = []
    for row in tqdm(qa_rows, desc=config_name):
        question = get_question(row)
        gt_ids = ground_truth_chunk_ids(row)

        t0 = time.perf_counter()
        te0 = time.perf_counter()
        qvec = model.encode(['query: ' + question], normalize_embeddings=True, show_progress_bar=False)[0].astype('float32').tolist()
        embed_ms = (time.perf_counter() - te0) * 1000

        ts0 = time.perf_counter()
        hits = store.search(qvec, limit=RETRIEVE_K)
        search_ms = (time.perf_counter() - ts0) * 1000
        total_ms = (time.perf_counter() - t0) * 1000

        hit_chunk_ids = [str(h.payload.get('chunk_id') or h.point_id) for h in hits]
        case = {
            'qa_id': row.get('qa_id') or row.get('id'),
            'question': question,
            'ground_truth_chunk_ids': sorted(gt_ids),
            'retrieved_chunk_ids': hit_chunk_ids,
            'embed_ms': embed_ms,
            'search_ms': search_ms,
            'total_ms': total_ms,
            'category': row.get('category'),
            'difficulty': row.get('difficulty'),
            'answer_type': row.get('answer_type'),
        }
        case.update(metrics_for_case(hit_chunk_ids, gt_ids, TOP_K_LIST))
        cases.append(case)

    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    run_dir = OUT_DIR / config_name / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    with (run_dir / 'retrieval_cases.jsonl').open('w', encoding='utf-8') as f:
        for case in cases:
            f.write(json.dumps(case, ensure_ascii=False) + '\n')

    summary = aggregate_cases(cases)
    breakdowns = {
        'by_category': aggregate_by_field(cases, 'category'),
        'by_difficulty': aggregate_by_field(cases, 'difficulty'),
        'by_answer_type': aggregate_by_field(cases, 'answer_type'),
    }
    latency = latency_summary(summary)
    manifest = {
        'run_id': run_id,
        'config_name': config_name,
        'benchmark_path': str(QA_PATH),
        'index_path': str(index_dir),
        'model': MODEL_NAME,
        'top_k': TOP_K_LIST,
        'retrieve_k': RETRIEVE_K,
        'sample_limit': SAMPLE_LIMIT,
        'n_evaluated': len(cases),
        'timestamp': run_id,
        'notes': 'Dense embedding ablation; FAISS + SQLite payload cache. Retrieval-only run; E2E generation metrics are handled by the E2E runner.',
    }
    (run_dir / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    (run_dir / 'retrieval_metrics.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    (run_dir / 'retrieval_breakdown.json').write_text(json.dumps(breakdowns, ensure_ascii=False, indent=2), encoding='utf-8')
    (run_dir / 'latency.json').write_text(json.dumps(latency, ensure_ascii=False, indent=2), encoding='utf-8')
    write_report(run_dir, config_name, summary, breakdowns, manifest)

    store.close()
    print('saved:', run_dir)
    print(summary)
    return summary

summaries = {}
for config_name, index_dir in RUNTIME_INDEX_DIRS.items():
    summaries[config_name] = run_config(config_name, index_dir)

summary_df = pd.DataFrame.from_dict(summaries, orient='index')
summary_df


Embed-ChunkMeta-Dense
index_dir: /kaggle/working/faiss_indexes/Embed-ChunkMeta-Dense
Reusing SQLite payload cache: payload_cache.sqlite
FAISS index loaded in 38.75s
vectors: 1513376


Embed-ChunkMeta-Dense:   0%|          | 0/500 [00:00<?, ?it/s]

FAISS search: 0.729s | payload batch load: 0.122s | inspected: 10
FAISS search: 0.729s | payload batch load: 0.023s | inspected: 10
FAISS search: 0.711s | payload batch load: 0.053s | inspected: 10
FAISS search: 0.748s | payload batch load: 0.012s | inspected: 10
FAISS search: 0.716s | payload batch load: 0.081s | inspected: 10
FAISS search: 0.715s | payload batch load: 0.016s | inspected: 10
FAISS search: 0.722s | payload batch load: 0.025s | inspected: 10
FAISS search: 0.709s | payload batch load: 0.024s | inspected: 10
FAISS search: 0.704s | payload batch load: 0.056s | inspected: 10
FAISS search: 0.716s | payload batch load: 0.029s | inspected: 10
FAISS search: 0.705s | payload batch load: 0.036s | inspected: 10
FAISS search: 0.712s | payload batch load: 0.021s | inspected: 10
FAISS search: 0.707s | payload batch load: 0.040s | inspected: 10
FAISS search: 0.711s | payload batch load: 0.024s | inspected: 10
FAISS search: 0.735s | payload batch load: 0.028s | inspected: 10
FAISS sear

Embed-ChunkOnly-Dense:   0%|          | 0/500 [00:00<?, ?it/s]

FAISS search: 0.733s | payload batch load: 0.026s | inspected: 10
FAISS search: 0.719s | payload batch load: 0.013s | inspected: 10
FAISS search: 0.739s | payload batch load: 0.015s | inspected: 10
FAISS search: 0.733s | payload batch load: 0.011s | inspected: 10
FAISS search: 0.732s | payload batch load: 0.012s | inspected: 10
FAISS search: 0.713s | payload batch load: 0.008s | inspected: 10
FAISS search: 0.726s | payload batch load: 0.015s | inspected: 10
FAISS search: 0.734s | payload batch load: 0.007s | inspected: 10
FAISS search: 0.723s | payload batch load: 0.003s | inspected: 10
FAISS search: 0.721s | payload batch load: 0.001s | inspected: 10
FAISS search: 0.741s | payload batch load: 0.009s | inspected: 10
FAISS search: 0.730s | payload batch load: 0.001s | inspected: 10
FAISS search: 0.726s | payload batch load: 0.018s | inspected: 10
FAISS search: 0.725s | payload batch load: 0.009s | inspected: 10
FAISS search: 0.731s | payload batch load: 0.015s | inspected: 10
FAISS sear

,n_cases,mrr@10,ndcg@1,ndcg@10,ndcg@5,recall@1,recall@10,recall@5,embed_ms_mean,embed_ms_median,search_ms_mean,search_ms_median,total_ms_mean,total_ms_median
Embed-ChunkMeta-Dense,500,0.226680,0.1025,0.281304,0.240365,0.090208,0.485458,0.367792,34.796549,32.084547,734.468799,732.081123,769.267953,764.940490
Embed-ChunkOnly-Dense,500,0.286126,0.1650,0.322133,0.290052,0.148625,0.484667,0.390292,33.253158,32.037683,727.842454,724.999917,761.098076,758.998998


In [9]:
summary_csv = OUT_DIR / 'embedding_ablation_summary.csv'
summary_df.to_csv(summary_csv, encoding='utf-8-sig')
print('summary:', summary_csv)
summary_df

summary: /kaggle/working/evaluation_runs/dense_embedding_ablation/embedding_ablation_summary.csv


,n_cases,mrr@10,ndcg@1,ndcg@10,ndcg@5,recall@1,recall@10,recall@5,embed_ms_mean,embed_ms_median,search_ms_mean,search_ms_median,total_ms_mean,total_ms_median
Embed-ChunkMeta-Dense,500,0.226680,0.1025,0.281304,0.240365,0.090208,0.485458,0.367792,34.796549,32.084547,734.468799,732.081123,769.267953,764.940490
Embed-ChunkOnly-Dense,500,0.286126,0.1650,0.322133,0.290052,0.148625,0.484667,0.390292,33.253158,32.037683,727.842454,724.999917,761.098076,758.998998
